# Week 13 guided practice

CML3112. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and upload engineering_data.csv into Colab with the Files panel before running the cells. The target in this notebook is cooling load in kW.

## Today’s learning

*PowerPoint slide 2*

Hold a flexible model back with a penalty and predict from the nearest neighbours.
Compare five candidates fairly.
Make the project run for someone else, from a clean start.

You can say what regularisation, alpha and k mean, and read a table of candidates.
Another learner can follow your written instructions and reproduce your stated result.

## Preparing the rows once more

*PowerPoint slide 4*

The same cleaning as Weeks 8 to 12: drop copies, keep valid rows.
E1 to E4 train, E5 validates, E6 stays sealed.
X holds the two inputs. y holds the target.

Every experiment today uses these four tables.

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
raw = pd.read_csv("engineering_data.csv")
data = raw.drop_duplicates("record_id")
data = data[data["target"].notna()
            & data["load_pct"].between(0, 100)].copy()
groups = ["E1", "E2", "E3", "E4"]
train = data[data["equipment_id"].isin(groups)]
valid = data[data["equipment_id"] == "E5"]
features = ["load_pct", "ambient_c"]
X_train, y_train = train[features], train["target"]
X_valid, y_valid = valid[features], valid["target"]
print(len(train), len(valid))

**Example output**

```text
58 15
```

## A flexible curve can chase the training rows, and a penalty calms it

*PowerPoint slide 5*

Same degree-6 curve on load alone. Left: no penalty. Right: penalty alpha = 1.

## Ridge adds a price for large coefficients

*PowerPoint slide 6*

loss = sum of squared misses
+ alpha × (sum of squared coefficients)

alpha = 0 means no penalty. The method is ordinary least squares.
A larger alpha shrinks the coefficients toward zero and calms the curve.
We choose alpha with the validation rows, in the way that we chose tree depth.
The penalty treats all coefficients alike, so the inputs are scaled first. That is why StandardScaler sits in the pipeline.

## Three curve models that differ only in the penalty

*PowerPoint slide 7*

PolynomialFeatures(6) adds load squared up to load to the power of 6.
StandardScaler puts those columns on a common ruler.
The last step is the estimator: no penalty, alpha 1 or alpha 100.

A dictionary keeps the three models side by side. Nothing is fitted yet.

In [ ]:
from sklearn.preprocessing import (
    PolynomialFeatures, StandardScaler)
from sklearn.linear_model import LinearRegression, Ridge
def curve_model(estimator):
    return make_pipeline(
        PolynomialFeatures(6, include_bias=False),
        StandardScaler(), estimator)
models = {
    "none": curve_model(LinearRegression()),
    "alpha 1": curve_model(Ridge(alpha=1)),
    "alpha 100": curve_model(Ridge(alpha=100)),
}

## Fitting the three curves and scoring them

*PowerPoint slide 8*

X_train[["load_pct"]] keeps the load column only, as a table.
Each model is fitted on the training rows and scored on both sets.
Errors are in kW.

A stronger penalty lowers the fit to the training rows. The validation rows say whether that helps.

In [ ]:
load_train = X_train[["load_pct"]]
load_valid = X_valid[["load_pct"]]
print("model train valid")
for name, model in models.items():
    model.fit(load_train, y_train)
    train_pred = model.predict(load_train)
    valid_pred = model.predict(load_valid)
    train_mae = mean_absolute_error(y_train, train_pred)
    valid_mae = mean_absolute_error(y_valid, valid_pred)
    print(name, round(train_mae, 2), round(valid_mae, 2))

**Example output**

```text
model train valid
none 2.91 3.32
alpha 1 3.01 3.05
alpha 100 3.97 4.68
```

## What the penalty does to the coefficients and the errors

*PowerPoint slide 9*

A plain straight line on load alone: validation MAE 3.12 kW.
The Week 8 line on load and ambient temperature: 0.74 kW.
In scikit-learn the coefficients are in model[-1].coef_, the last step of the pipeline.

## A prediction from the three nearest training rows

*PowerPoint slide 11*

The star is the validation row with record_id 5. Its 3 nearest training rows (scaled inputs) have targets 44.323, 47.958 and 50.268.

## Trying three values of k

*PowerPoint slide 12*

n_neighbors is k, the number of neighbours that vote.
The imputer fills blanks and the scaler puts inputs on a common ruler before distances are measured.
Errors are in kW, on the validation rows.

fit stores the training rows. predict does the neighbour search.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
print("k valid")
for k in [1, 3, 20]:
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=k))
    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)
    valid_mae = mean_absolute_error(y_valid, valid_pred)
    print(k, round(valid_mae, 2))

**Example output**

```text
k valid
1 1.69
3 1.16
20 3.72
```

## Choosing k, and the price of waiting until prediction time

*PowerPoint slide 13*

CHOOSING k
Try several values and compare the validation error. Very small k chases noise. Very large k averages too much.
Here k = 3 gives the lowest error of the three we tried.

THE PRICE
KNN keeps every training row. Each prediction measures the distance to all of them, so it slows down as the training table grows.
A line or a tree answers in an instant once it is trained.

## Five candidates on the same validation rows

*PowerPoint slide 14*

All five were scored on the same validation rows.
The ridge row sees load only, so it is not a like-for-like comparison.
alpha and k were chosen on these rows, so the scores are slightly optimistic.

## Connection to the previous lesson

*PowerPoint slide 17*

The models and charts already exist.

The remaining task is to make the evidence understandable and runnable from the supplied files.

## The continuing pandas workflow

*PowerPoint slide 18*

Load CSV files with pd.read_csv. Keep raw data unchanged and use named columns throughout the investigation.

Use pandas for cleaning, summaries and model inputs. Return the published Python types at assessment function boundaries.

## Relative data paths and saved outputs

*PowerPoint slide 19*

pd.read_csv("data/raw/engineering_data.csv") reads a file relative to the notebook’s working folder. State that folder in the README.

clean.to_csv("cleaned_data.csv", index=False) exports a table without adding pandas row labels as a new column.

## Why this week’s tools are needed

*PowerPoint slide 20*

A project is ready for handover when its files and written instructions reproduce its results in a fresh session.

Check each claim against the results in your notebook.

## Reproducibility and Dependency

*PowerPoint slide 21*

Reproducibility

Another person can use the stated data, environment and steps to check the result. Terminology varies; state the test you used.

Dependency

A package your code needs. Record versions actually used, rather than guessing them.

## Relative path and README

*PowerPoint slide 22*

Relative path

A location interpreted from the current working folder. It is not always relative to the notebook file itself.

The project’s instructions: question, files, data source, environment, run order, results and limitations.

## A useful README

*PowerPoint slide 23*

01

State the engineering question and target unit.

02

List the supplied data and required software versions.

03

Give the order for running the notebook from a fresh session.

04

State limitations and each member’s contribution.

## Someone else needs to reproduce the result

*PowerPoint slide 24*

Which instructions would a new user need?

## A reproducible notebook starts fresh

*PowerPoint slide 25*

The saved outputs must agree with a fresh run in the stated order.

## Saved outputs can hide missing steps

*PowerPoint slide 26*

01

A notebook can display an old answer after its inputs have changed.

02

A fresh session begins without the old variables.

03

Restarting and running all tests whether the stated order creates the result.

## A path names where a file is found

*PowerPoint slide 27*

01

A relative path is interpreted from the current working folder.

02

Package versions describe the software used to run the code.

03

The README must give both run instructions and data-access instructions.

## Read the path example

*PowerPoint slide 28*

Path and cwd

Path is a tool imported from Python’s pathlib module. Path.cwd() returns the folder where Python is currently running.

name

The .name property gives the last folder component as text. It depends on your runtime. A different folder name is expected on another computer.

## Tell another person what the project needs

*PowerPoint slide 29*

requirements.txt and versions

This text file lists required Python packages and their versions. Record the versions used in the working environment; test them with the supplied run instructions.

Data source, permission and seed

Name the data source and any licence or permission for reuse. Record random seeds used in modelling. A seed supports repeatability; it does not repair data leakage.

## Recording the versions you used

*PowerPoint slide 30*

Each line prints the version of one tool.
Copy the four results into requirements.txt and the README.
Write them as pandas==X.Y.Z, one tool on each line, with the numbers your own run printed.

Install later with: pip install -r requirements.txt

In [ ]:
import sys, pandas, numpy, sklearn
print(sys.version.split()[0])
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("scikit-learn", sklearn.__version__)

## A fresh start in Colab

*PowerPoint slide 31*

BEFORE THE CHECK
Upload your data file to the Colab Files panel, or mount Drive.
Put the file where the README says, for example data/raw/engineering_data.csv.

THE CHECK
Choose Runtime, then Restart session, then Run all.
Disconnect and delete runtime wipes uploaded files, so you upload them again before the check.

## Run checks and contribution evidence

*PowerPoint slide 32*

Independent run check

Restart the session and follow your written instructions without relying on earlier variables. Record the first failure and its repair.

Contribution evidence

A record of each person’s work, tied to files, commits or explanations. It does not require making a private repository public.

## Useful feedback identifies an exact repair

*PowerPoint slide 33*

01

Vague feedback: "It does not work."

02

Useful feedback: "Step 2 names data.csv, but the supplied file is readings.csv."

03

The repair is to make the instructions and actual filename agree.

## Revision example and answer

*PowerPoint slide 34*

Check: restart the notebook runtime and follow your README exactly, without using remembered variables.

If Step 2 names data.csv but the file is readings.csv, correct the filename in the instructions. Run again from the beginning and record the successful result.

## Block 2 closes after today

*PowerPoint slide 35*

No new weekly assignment.

The Block 2 pack, A10 and A11 for 40 marks, is due Monday 7 December 2026, 23:59 Kampala time.
The Block 2 quiz covers Weeks 11, 12 and 13. It opens after today's class.
Next week is Practical Test 2. Rehearse saving and submitting your test notebook, and revise the published function interfaces.

## A reproducible handover

*PowerPoint slide 36*

01

An absolute path from your own computer will usually fail elsewhere.

02

A relative path starts from the current working folder.

03

Record dependencies and actual versions, and fix randomness where appropriate; identical results across all machines are not guaranteed by a seed alone.

04

Captions should state a supported finding and its scope.

## A reproducible handover continued

*PowerPoint slide 37*

Remove conflicting cell assignments and verify a fresh run before submission.